# 2.5.自动微分

正如 [<span style="color: #ffbf00; font-weight: bold;"> 2.4节：微积分 </span>](./02.04_calculus.ipynb) 中所说，求导是几乎所有深度学习优化算法的关键步骤。虽然求导的计算很简单，只需要一些基本的微积分。但对于复杂的模型，手工进行更新是一件很痛苦的事情（而且经常容易出错）。

深度学习框架通过自动计算导数，即*自动微分*（automatic differentiation）来加快求导。实际中，根据设计好的模型，系统会构建一个*计算图*（computational graph），来跟踪计算是哪些数据通过哪些操作组合起来产生输出。自动微分使系统能够随后反向传播梯度。这里，*反向传播*（backpropagate）意味着跟踪整个计算图，填充关于每个参数的偏导数。

需要注意的是，PyPTO目前的版本不内置自动微分功能，因而本节中将保留原文，使用PyTorch介绍自动微分。

---

## 环境准备

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"

import pypto
import torch

---

## 2.5.1.一个简单的例子

作为一个演示例子，假设我们想对函数$y=2\mathbf{x}^{\top}\mathbf{x}$关于列向量$\mathbf{x}$求导。首先，我们创建变量`x`并为其分配一个初始值。

In [ ]:
x = torch.arange(4.0)
print(x)

tensor([0., 1., 2., 3.])


在我们计算$y$关于$\mathbf{x}$的梯度之前，需要一个地方来存储梯度。重要的是，我们不会在每次对一个参数求导时都分配新的内存。因为我们经常会成千上万次地更新相同的参数，每次都分配新的内存可能很快就会将内存耗尽。注意，一个标量函数关于向量$\mathbf{x}$的梯度是向量，并且与$\mathbf{x}$具有相同的形状。

In [ ]:
x.requires_grad_(True)  # 等价于x=torch.arange(4.0,requires_grad=True)
print(x.grad)  # 默认值是None

None


现在计算$y$。


In [ ]:
y = 2 * torch.dot(x, x)
print(y)

tensor(28., grad_fn=<MulBackward0>)


`x`是一个长度为4的向量，计算`x`和`x`的点积，得到了我们赋值给`y`的标量输出。接下来，通过调用反向传播函数来自动计算`y`关于`x`每个分量的梯度，并打印这些梯度。


In [ ]:
y.backward()
print(x.grad)

tensor([ 0.,  4.,  8., 12.])


函数$y=2\mathbf{x}^{\top}\mathbf{x}$关于$\mathbf{x}$的梯度应为$4\mathbf{x}$。让我们快速验证这个梯度是否计算正确。


In [ ]:
print(x.grad == 4 * x)

tensor([True, True, True, True])


现在计算`x`的另一个函数。


In [ ]:
# 在默认情况下，PyTorch会累积梯度，我们需要清除之前的值
x.grad.zero_()
y = x.sum()
y.backward()
print(x.grad)

tensor([1., 1., 1., 1.])


---

## 2.5.2.非标量变量的反向传播

当`y`不是标量时，向量`y`关于向量`x`的导数的最自然解释是一个矩阵。对于高阶和高维的`y`和`x`，求导的结果可以是一个高阶张量。

然而，虽然这些更奇特的对象确实出现在高级机器学习中（包括深度学习中），但当调用向量的反向计算时，我们通常会试图计算一批训练样本中每个组成部分的损失函数的导数。这里，我们的目的不是计算微分矩阵，而是单独计算批量中每个样本的偏导数之和。

In [ ]:
# 对非标量调用backward需要传入一个gradient参数，该参数指定微分函数关于self的梯度。
# 本例只想求偏导数的和，所以传递一个1的梯度是合适的
x.grad.zero_()
y = x * x
# 等价于y.backward(torch.ones(len(x)))
y.sum().backward()
print(x.grad)

tensor([0., 2., 4., 6.])


---

## 2.5.3.分离计算

有时，我们希望将某些计算移动到记录的计算图之外。例如，假设`y`是作为`x`的函数计算的，而`z`则是作为`y`和`x`的函数计算的。想象一下，我们想计算`z`关于`x`的梯度，但由于某种原因，希望将`y`视为一个常数，并且只考虑到`x`在`y`被计算后发挥的作用。

这里可以分离`y`来返回一个新变量`u`，该变量与`y`具有相同的值，但丢弃计算图中如何计算`y`的任何信息。换句话说，梯度不会向后流经`u`到`x`。因此，下面的反向传播函数计算`z=u*x`关于`x`的偏导数，同时将`u`作为常数处理，而不是`z=x*x*x`关于`x`的偏导数。


In [ ]:
x.grad.zero_()
y = x * x
u = y.detach()
z = u * x

z.sum().backward()
print(x.grad == u)

tensor([True, True, True, True])


由于记录了`y`的计算结果，我们可以随后在`y`上调用反向传播，得到`y=x*x`关于`x`的导数，即`2*x`。


In [ ]:
x.grad.zero_()
y.sum().backward()
print(x.grad == 2 * x)

tensor([True, True, True, True])


---

## 2.5.4.Python控制流的梯度计算

使用自动微分的一个好处是：即使构建函数的计算图需要通过Python控制流（例如，条件、循环或任意函数调用），我们仍然可以计算得到的变量的梯度。在下面的代码中，`while`循环的迭代次数和`if`语句的结果都取决于输入`a`的值。


In [ ]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

让我们计算梯度。


In [ ]:
a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()

我们现在可以分析上面定义的`f`函数。请注意，它在其输入`a`中是分段线性的。换言之，对于任何`a`，存在某个常量标量`k`，使得`f(a)=k*a`，其中`k`的值取决于输入`a`，因此可以用`d/a`验证梯度是否正确。


In [ ]:
print(a.grad == d / a)

tensor(True)


---

## 2.5.5.PyPTO算子接入PyTorch自动微分

PyPTO目前的版本不内置自动微分功能，但它可以接入PyTorch的自动微分框架，使自定义的PyPTO算子能够参与梯度计算。一个实用的接入方式是：

1. 用 `@pypto.frontend.jit` 实现 PyPTO kernel：一般至少包含一个前向 kernel；如果希望反向也在 NPU 上高效执行，也可以再写一个反向 kernel；
2. 用 `torch.autograd.Function` 把这些 kernel 包起来；
3. 在 `backward` 中给出梯度公式，可以调用 PyTorch 自带算子，也可以继续调用反向 PyPTO kernel。

下面用一个简单的平方算子 `y=x^2` 演示这种接入方式。它的梯度很容易验证：
$$
\frac{dy}{dx}=2x
$$
如果再对 `y.sum()` 反向传播，则有：
$$
\frac{d\sum y}{dx}=2x
$$

In [ ]:
@pypto.frontend.jit()
def square_kernel(
    x: pypto.Tensor([...], pypto.DT_FP32),
    out: pypto.Tensor([...], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(1, 4, 1, 64)
    out[:] = x * x

接下来把这个 PyPTO kernel 包装成一个 PyTorch 自定义算子。把自定义算子接入 PyTorch 自动微分的关键是 `torch.autograd.Function` 这个类。一般来说，继承之后，你需要定义两个方法：

1. `forward(ctx, *inputs)`：执行前向计算，返回输出。
2. `backward(ctx, grad_out)`：执行反向计算，返回下游梯度。`grad_out` 是上游传回的梯度，返回值必须与 `forward` 的输入一一对应。

在两个方法里面，有几个需要注意的操作。一般来说，反向计算都需要使用前向操作的输入输出或中间产物。所以我们需要在前向中调用 `ctx.save_for_backward(...)` 保存张量到上下文中，再在反向中调用 `ctx.saved_tensors` 解包前向保存的张量。

需要注意的是，在 PyTorch 中进行 `reshape` 等一些操作的时候，可能会导致 Tensor 非连续。由于 PyPTO 规定进入 kernel 的 Tensor 必须连续，我们一般需要先对需要进入 kernel 的 Tensor 使用 `.contiguous()` 取连续。

了解以上几点后，再回头看下面的 `PyPTOSquare` 类：

In [ ]:
class PyPTOSquare(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        x_c = x.contiguous()
        ctx.save_for_backward(x_c)
        out = torch.empty_like(x_c)
        square_kernel(x_c, out)
        return out

    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        grad_x = grad_out * 2 * x
        return grad_x

def pypto_square(x):
    return PyPTOSquare.apply(x)

定义完成之后，我们先执行前向传播。

In [ ]:
x = torch.arange(4.0, device='npu:0', requires_grad=True)
y = pypto_square(x)
print(y)

tensor([0., 1., 4., 9.], device='npu:0', grad_fn=<PyPTOSquareBackward>)


<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
        <li style="margin: 0 0 8px 0;">创建 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">tensor</code> 时添加 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">device='npu:0'</code> 参数的原因是：</li>
        <p style="margin: 0;">参与 NPU 上运行的 PyPTO 算子的 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">tensor</code> 通常需要位于 NPU 中，这一般是通过指定 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">device</code> 参数来完成的。这里使用 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">device='npu:0'</code> 来指定，在大多数环境下可以正常运行（注意这并 <span class="em-not" style="font-weight: 600; color: #b91c1c; background-color: #fef2f2; padding: 0 3px; border-radius: 2px;">不</span> 是严谨的写法，只是出于简洁需要而这么写）。除此之外，也可以对 CPU 上的 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">tensor</code> 使用其 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">.npu()</code> 方法。</p>
    </ul>
  </div>
</details>

随后，执行反向传播。

In [ ]:
y.sum().backward()
print(x.grad)

tensor([0., 2., 4., 6.], device='npu:0')


PyTorch的反向传播框架输出了我们想要的结果。

In [ ]:
print(x.grad == 2 * x)

tensor([True, True, True, True], device='npu:0')


从结果可以看到，虽然前向计算是通过 PyPTO kernel 完成的，但只要我们把它包进 `torch.autograd.Function` 并提供 `backward`，它就能够正常接入 PyTorch 的自动微分体系。

这种方式适合两类场景：

1. 前向希望调用自定义的 PyPTO 高性能算子；
2. 反向传播的求导公式比较清楚，可以直接手工写出来。

需要注意的是，这里自动微分能力仍然来自 PyTorch，而不是 PyPTO 内部自动生成的反向图。也就是说，PyPTO 负责算子执行，但梯度规则与算子之间的连接关系由我们显式给出。

---

## 2.5.6.小结

* 深度学习框架可以自动计算导数：我们首先将梯度附加到想要对其计算偏导数的变量上，然后记录目标值的计算，执行它的反向传播函数，并访问得到的梯度。

---

## 2.5.7.练习

1. 为什么计算二阶导数比一阶导数的开销要更大？
2. 在运行反向传播函数之后，立即再次运行它，看看会发生什么。
3. 在控制流的例子中，我们计算`d`关于`a`的导数，如果将变量`a`更改为随机向量或矩阵，会发生什么？
4. 重新设计一个求控制流梯度的例子，运行并分析结果。
5. 使$f(x)=\sin(x)$，绘制$f(x)$和$\frac{df(x)}{dx}$的图像，其中后者不使用$f'(x)=\cos(x)$。

参考答案见 [answers/02.05_reference_answer](./answers/02.05_reference_answer.ipynb)。
